# Tema 3.1 · Extracción de características clásicas

En este notebook convertimos imágenes en **vectores de números** que describen su color, su forma y su textura, y vemos cómo se comparan.

> Recordatorio: una buena característica es **robusta**, **discriminativa** y **eficaz**... y lo que es «bueno» depende de la aplicación.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from scipy import stats, ndimage as ndi
from skimage import data, color, exposure, feature, filters, measure, morphology, segmentation, transform, draw, util
from skimage.filters.rank import entropy as rank_entropy

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
plt.rcParams['figure.dpi'] = 100
pd.set_option('display.precision', 3)

def show(images, titles=None, cmap='gray', size=3.2):
    """Muestra una lista de imágenes en una fila."""
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(size * n, size))
    axes = np.atleast_1d(axes)
    for i, (ax, im) in enumerate(zip(axes, images)):
        ax.imshow(im, cmap=None if im.ndim == 3 else cmap)
        ax.set_title(titles[i] if titles else '')
        ax.axis('off')
    plt.tight_layout()
    plt.show()

warnings.filterwarnings('ignore', category=UserWarning)
print('OpenCV', cv2.__version__)

## 1. Características espectrales

Son **globales**: describen la distribución de valores de los píxeles en toda la imagen, sin importar dónde están.

### 1.1 Histogramas por canal

In [ ]:
imgs = {'astronauta': data.astronaut(), 'gato': data.chelsea(), 'café': data.coffee()}
show(list(imgs.values()), list(imgs.keys()))

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, (nombre, im) in zip(axes, imgs.items()):
    for c, col in enumerate(['red', 'green', 'blue']):
        h, _ = np.histogram(im[..., c], bins=64, range=(0, 256), density=True)
        ax.plot(np.linspace(0, 255, 64), h, color=col)
    ax.set_title(nombre); ax.set_xlabel('intensidad')
plt.tight_layout(); plt.show()

### 1.2 Momentos estadísticos

Implementamos las fórmulas de las diapositivas (μ, varianza σ, sesgo y curtosis) y las comprobamos con `scipy.stats`.
Un histograma de 64 bins por canal da 192 números; los **4 momentos por canal** resumen la imagen en sólo 12.

In [ ]:
def momentos(x):
    x = x.astype(float).ravel()
    mu = x.mean()
    var = ((x - mu) ** 2).mean()            # σ en la diapositiva (varianza)
    sesgo = ((x - mu) ** 3).mean() / var ** 1.5
    curt = ((x - mu) ** 4).mean() / var ** 2
    return mu, var, sesgo, curt

x = imgs['gato'][..., 0]
mu, var, s, k = momentos(x)
print(f'manual: μ={mu:.2f}  var={var:.2f}  sesgo={s:.3f}  curtosis={k:.3f}')
print(f'scipy : sesgo={stats.skew(x.ravel()):.3f}  curtosis={stats.kurtosis(x.ravel(), fisher=False):.3f}')

filas = []
for nombre, im in imgs.items():
    for c, canal in enumerate('RGB'):
        filas.append([nombre, canal, *momentos(im[..., c])])
tabla = pd.DataFrame(filas, columns=['imagen', 'canal', 'media', 'varianza', 'sesgo', 'curtosis'])
display(tabla.pivot(index='imagen', columns='canal', values=['media', 'sesgo']))

### 1.3 Espacios de color y robustez a la iluminación

En RGB, el brillo está mezclado con el color. En HSV el **tono (H)** separa «qué color» de «cuánta luz»; en L\*a\*b\* la luminosidad va en L\* y el color en a\*, b\*.

Experimento: oscurecemos la imagen un 40 % y medimos cuánto cambia el histograma en cada representación con la distancia χ² (la misma que en el bloque 3.3).

In [ ]:
def hist_norm(canal, bins=32, rango=(0, 1)):
    h, _ = np.histogram(canal, bins=bins, range=rango)
    return h / h.sum()

def chi2(h, g, eps=1e-10):
    return 0.5 * np.sum((h - g) ** 2 / (h + g + eps))

im = util.img_as_float(data.chelsea())
oscura = np.clip(im * 0.6, 0, 1)
show([im, oscura], ['original', '60 % de brillo'])

hsv_a, hsv_b = color.rgb2hsv(im), color.rgb2hsv(oscura)
lab_a, lab_b = color.rgb2lab(im), color.rgb2lab(oscura)
res = {
    'R (RGB)': chi2(hist_norm(im[..., 0]), hist_norm(oscura[..., 0])),
    'H (HSV)': chi2(hist_norm(hsv_a[..., 0]), hist_norm(hsv_b[..., 0])),
    'V (HSV)': chi2(hist_norm(hsv_a[..., 2]), hist_norm(hsv_b[..., 2])),
    'a* (Lab)': chi2(hist_norm(lab_a[..., 1], rango=(-60, 60)), hist_norm(lab_b[..., 1], rango=(-60, 60))),
    'L* (Lab)': chi2(hist_norm(lab_a[..., 0], rango=(0, 100)), hist_norm(lab_b[..., 0], rango=(0, 100))),
}
for k, v in res.items():
    print(f'{k:9s}  χ² = {v:.3f}')

**Lectura.** El tono H no cambia (χ² ≈ 0) porque escalar R, G y B por el mismo factor no altera su proporción; V y L\* absorben el cambio de iluminación. a\* cambia algo porque la conversión a L\*a\*b\* no es lineal. Por eso en muchos proyectos se describe el color con H (o con a\*, b\*) y se ignora el brillo.

## 2. Características de forma

Primero hay que **segmentar** los objetos. Usamos la imagen de monedas: gradiente de Sobel + *watershed* con marcadores (fondo oscuro, monedas claras).

In [ ]:
monedas = data.coins()
grad = filters.sobel(monedas)
marcadores = np.zeros_like(monedas)
marcadores[monedas < 30] = 1
marcadores[monedas > 150] = 2
seg = segmentation.watershed(grad, marcadores) == 2
seg = ndi.binary_fill_holes(seg)
seg = morphology.remove_small_objects(seg, 300)
etiquetas = measure.label(seg)
print('Objetos encontrados:', etiquetas.max())
show([monedas, seg, color.label2rgb(etiquetas, monedas, bg_label=0)], ['original', 'máscara', 'instancias'])

In [ ]:
props = measure.regionprops_table(etiquetas, properties=('label', 'area', 'perimeter', 'major_axis_length',
                                                          'minor_axis_length', 'eccentricity', 'centroid'))
df = pd.DataFrame(props)
df['compacidad'] = 4 * np.pi * df['area'] / df['perimeter'] ** 2
display(df.sort_values('area', ascending=False).head(8))

La **compacidad** vale 1 para un círculo perfecto y baja cuanto más alargada o irregular es la forma. Comprobémoslo con figuras sintéticas (el perímetro digital no es exacto, así que el círculo no da 1,000 exactamente).

In [ ]:
def forma(mask):
    p = measure.regionprops(mask.astype(int))[0]
    return p.area, p.perimeter, 4 * np.pi * p.area / p.perimeter ** 2

figs = {}
m = np.zeros((200, 200), bool); rr, cc = draw.disk((100, 100), 70); m[rr, cc] = True; figs['círculo'] = m
m = np.zeros((200, 200), bool); m[40:160, 40:160] = True; figs['cuadrado'] = m
m = np.zeros((200, 200), bool); m[85:115, 10:190] = True; figs['barra'] = m
ang = np.linspace(0, 2 * np.pi, 11)[:-1]
r = np.where(np.arange(10) % 2 == 0, 90, 35)
m = np.zeros((200, 200), bool); rr, cc = draw.polygon(100 + r * np.sin(ang), 100 + r * np.cos(ang)); m[rr, cc] = True; figs['estrella'] = m

show(list(figs.values()), [f'{k}\ncompacidad={forma(v)[2]:.2f}' for k, v in figs.items()])

## 3. Características de textura

### 3.1 Matriz de co-ocurrencia (GLCM) a mano

Construimos la GLCM de una imagen 4×4 con 4 niveles de gris y la relación espacial (1, 0): vecino **a la derecha**. Después la hacemos **simétrica** (sumando su traspuesta) y la normalizamos para que sea una **probabilidad**, como en las diapositivas. Finalmente lo comprobamos con `skimage.feature.graycomatrix`.

In [ ]:
I = np.array([[0, 0, 1, 1],
              [0, 0, 1, 1],
              [0, 2, 2, 2],
              [2, 2, 3, 3]])
Ng = 4
C = np.zeros((Ng, Ng), int)
for y in range(I.shape[0]):
    for x in range(I.shape[1] - 1):          # Δx = 1, Δy = 0
        C[I[y, x], I[y, x + 1]] += 1
print('Co-ocurrencias (1,0):\n', C)
S = C + C.T
print('\nSimétrica:\n', S)
P = S / S.sum()
print('\nProbabilidad p(i,j):\n', P.round(3))

G = feature.graycomatrix(I.astype(np.uint8), distances=[1], angles=[0], levels=Ng, symmetric=True, normed=True)[:, :, 0, 0]
print('\n¿Coincide con skimage?', np.allclose(P, G))

### 3.2 Estadísticos de Haralick

Implementamos contraste, homogeneidad, entropía y correlación con las fórmulas de las diapositivas y comparamos con `graycoprops` (que no incluye la entropía).

In [ ]:
def haralick(P):
    Ng = P.shape[0]
    i, j = np.meshgrid(np.arange(Ng), np.arange(Ng), indexing='ij')
    contraste = np.sum((i - j) ** 2 * P)
    homogeneidad = np.sum(P / (1 + (i - j) ** 2))
    nz = P[P > 0]
    entropia = -np.sum(nz * np.log2(nz))
    mu_i, mu_j = np.sum(i * P), np.sum(j * P)
    s_i = np.sqrt(np.sum((i - mu_i) ** 2 * P)); s_j = np.sqrt(np.sum((j - mu_j) ** 2 * P))
    correlacion = np.sum((i - mu_i) * (j - mu_j) * P) / (s_i * s_j)
    return dict(contraste=contraste, homogeneidad=homogeneidad, entropia=entropia, correlacion=correlacion)

print({k: round(float(v), 4) for k, v in haralick(P).items()})
G4 = feature.graycomatrix(I.astype(np.uint8), [1], [0], levels=4, symmetric=True, normed=True)
print({p: round(float(feature.graycoprops(G4, p)[0, 0]), 4) for p in ['contrast', 'homogeneity', 'correlation']})

Ahora con texturas reales. Recortamos 256×256 píxeles, cuantizamos a 32 niveles (con 256 niveles la matriz sería enorme y muy dispersa) y promediamos 4 orientaciones (0°, 45°, 90°, 135°) para ganar invariancia a la rotación.

In [ ]:
texturas = {'ladrillo': data.brick(), 'hierba': data.grass(), 'grava': data.gravel(), 'luna': data.moon()}

def recorte(im, s=256):
    return util.img_as_ubyte(transform.resize(im, (s, s), anti_aliasing=True))

def glcm_feats(im, niveles=32, d=1):
    q = (im.astype(int) * niveles // 256).astype(np.uint8)
    G = feature.graycomatrix(q, [d], [0, np.pi / 4, np.pi / 2, 3 * np.pi / 4], levels=niveles, symmetric=True, normed=True)
    P = G[:, :, 0, :].mean(axis=2)
    return haralick(P)

show([recorte(v) for v in texturas.values()], list(texturas.keys()))
display(pd.DataFrame({k: glcm_feats(recorte(v)) for k, v in texturas.items()}).T)

**Lectura.** El ladrillo, con bordes nítidos y regulares, tiene alto contraste; la luna, suave, tiene alta homogeneidad y correlación. Cuatro números ya separan texturas muy distintas.

### 3.3 Entropía local

La entropía calculada en una ventana deslizante produce un **mapa** de «desorden» local: alta en zonas texturadas, baja en zonas lisas.

In [ ]:
cam = data.camera()
ent = rank_entropy(cam, morphology.disk(5))
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(cam, cmap='gray'); ax[0].set_title('original'); ax[0].axis('off')
m = ax[1].imshow(ent, cmap='magma'); ax[1].set_title('entropía local (disco r=5)'); ax[1].axis('off')
plt.colorbar(m, ax=ax[1], fraction=0.046); plt.tight_layout(); plt.show()

### 3.4 Local Binary Patterns (LBP)

Cada píxel se compara con sus P vecinos: si el vecino es ≥ que el centro, aporta un 1. El histograma de códigos es el descriptor. Como sólo importa el **signo** de la diferencia, el LBP es **invariante a cambios monótonos de brillo**. Comprobémoslo:

In [ ]:
P_, R_ = 8, 1
im = recorte(texturas['ladrillo'])
lbp = feature.local_binary_pattern(im, P_, R_, method='uniform')
im2 = im.astype(np.int32) * 2 + 30          # transformación monótona lineal, exacta en enteros
lbp2 = feature.local_binary_pattern(im2, P_, R_, method='uniform')
print('Brillo ×2 + 30         → ¿LBP idéntico?', np.array_equal(lbp, lbp2))

im3 = (im / 255.0) ** 0.5                   # transformación monótona NO lineal (gamma)
lbp3 = feature.local_binary_pattern(im3, P_, R_, method='uniform')
print(f'Gamma 0,5              → {100 * np.mean(lbp == lbp3):.1f} % de píxeles con el mismo código')

im4 = np.round(im * 0.7 + 20).astype(np.uint8)   # lineal, pero redondeando a enteros
lbp4 = feature.local_binary_pattern(im4, P_, R_, method='uniform')
print(f'×0,7 + 20 redondeado   → {100 * np.mean(lbp == lbp4):.1f} % de píxeles con el mismo código')

def lbp_hist(im, P=8, R=1):
    l = feature.local_binary_pattern(im, P, R, method='uniform')
    h, _ = np.histogram(l, bins=np.arange(P + 3), density=True)
    return h

fig, axes = plt.subplots(1, 4, figsize=(14, 2.8), sharey=True)
for ax, (k, v) in zip(axes, texturas.items()):
    ax.bar(range(10), lbp_hist(recorte(v))); ax.set_title(k)
plt.suptitle('Histogramas LBP uniformes (P=8, R=1): 10 bins'); plt.tight_layout(); plt.show()

> Con una transformación lineal exacta el LBP es idéntico. Hay dos letras pequeñas: los vecinos diagonales se **interpolan**, y la interpolación no conserva el orden bajo transformaciones no lineales (gamma); y al **redondear** a enteros, dos niveles distintos pueden quedar iguales. Por eso en la práctica la invariancia es alta pero no total.

## 4. HOG: histogramas de gradientes orientados

In [ ]:
im = transform.resize(color.rgb2gray(data.astronaut()), (256, 256), anti_aliasing=True)
fd, hog_img = feature.hog(im, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2), visualize=True)
celdas = 256 // 8
bloques = celdas - 1
print('Longitud del vector HOG:', fd.shape[0])
print('Esperado: bloques² × (2·2 celdas) × 9 orientaciones =', bloques ** 2 * 4 * 9)
show([im, exposure.rescale_intensity(hog_img, in_range=(0, hog_img.max() / 3))], ['imagen', 'HOG'], size=4)

## 5. Puntos clave locales: SIFT y ORB

Las características **locales** detectan regiones de interés y describen su entorno. Comprobamos la invariancia de SIFT a rotación y escala: giramos 30° y reducimos al 70 % la imagen y buscamos correspondencias con el **test de razón de Lowe** (el vecino más cercano debe ser claramente mejor que el segundo).

In [ ]:
img1 = util.img_as_ubyte(color.rgb2gray(data.astronaut()))
h, w = img1.shape
M = cv2.getRotationMatrix2D((w / 2, h / 2), 30, 0.7)
img2 = cv2.warpAffine(img1, M, (w, h))

sift = cv2.SIFT_create()
kp1, d1 = sift.detectAndCompute(img1, None)
kp2, d2 = sift.detectAndCompute(img2, None)
print(f'SIFT: {len(kp1)} y {len(kp2)} puntos clave; descriptor de {d1.shape[1]} dimensiones')

bf = cv2.BFMatcher(cv2.NORM_L2)
buenos = [m for m, n in bf.knnMatch(d1, d2, k=2) if m.distance < 0.75 * n.distance]
print('Correspondencias que pasan el test de razón:', len(buenos))

vis = cv2.drawMatches(img1, kp1, img2, kp2, sorted(buenos, key=lambda m: m.distance)[:40], None,
                      flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
plt.figure(figsize=(12, 5)); plt.imshow(vis); plt.axis('off'); plt.title('40 mejores correspondencias SIFT'); plt.show()

¿Son correctas? Como conocemos la transformación real `M`, podemos comprobar cada correspondencia: un punto es correcto si, al transformarlo, cae a menos de 3 píxeles de su pareja. Repetimos con **ORB** (descriptor binario de 256 bits que se compara con la distancia de Hamming).

In [ ]:
def correctas(kpA, kpB, matches, M, tol=3):
    a = np.array([kpA[m.queryIdx].pt for m in matches])
    b = np.array([kpB[m.trainIdx].pt for m in matches])
    a_t = (M[:, :2] @ a.T).T + M[:, 2]
    return np.mean(np.linalg.norm(a_t - b, axis=1) < tol)

orb = cv2.ORB_create(nfeatures=2000)
ko1, o1 = orb.detectAndCompute(img1, None)
ko2, o2 = orb.detectAndCompute(img2, None)
buenos_orb = [m for m, n in cv2.BFMatcher(cv2.NORM_HAMMING).knnMatch(o1, o2, k=2) if m.distance < 0.75 * n.distance]

print(f'SIFT: {len(buenos):4d} correspondencias, {100 * correctas(kp1, kp2, buenos, M):.1f} % correctas')
print(f'ORB : {len(buenos_orb):4d} correspondencias, {100 * correctas(ko1, ko2, buenos_orb, M):.1f} % correctas')

## 6. Mini-experimento: ¿qué descriptor reconoce mejor la textura?

Construimos un pequeño dataset con 4 texturas. Para evitar **fuga de datos**, los recortes de entrenamiento salen de la mitad superior de cada imagen y los de prueba de la mitad inferior (si recortásemos de cualquier sitio, train y test compartirían píxeles y el resultado sería engañosamente bueno).

Comparamos tres representaciones con el mismo clasificador (k-NN, k=3):
1. Haralick (4 números)
2. Histograma LBP (10 números)
3. **Bolsa de palabras visuales** con descriptores SIFT densos y un vocabulario de K=32 palabras (k-means)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

rng = np.random.default_rng(0)
def recortes(im, n, mitad, s=64):
    im = recorte(im, 512)
    y0, y1 = (0, 256 - s) if mitad == 'sup' else (256, 512 - s)
    return [im[y:y + s, x:x + s] for y, x in zip(rng.integers(y0, y1, n), rng.integers(0, 512 - s, n))]

X_tr, y_tr, X_te, y_te = [], [], [], []
for c, im in enumerate(texturas.values()):
    tr, te = recortes(im, 40, 'sup'), recortes(im, 20, 'inf')
    X_tr += tr; y_tr += [c] * len(tr); X_te += te; y_te += [c] * len(te)
y_tr, y_te = np.array(y_tr), np.array(y_te)
print('train:', len(X_tr), 'recortes · test:', len(X_te), 'recortes')
show(X_tr[::40], list(texturas.keys()), size=2)

In [ ]:
# Descriptores SIFT densos: una rejilla de puntos clave fija sobre cada recorte
rejilla = [cv2.KeyPoint(float(x), float(y), 12) for y in range(8, 64, 8) for x in range(8, 64, 8)]
def sift_denso(im):
    _, d = sift.compute(im, rejilla)
    return d

K = 32
vocab = KMeans(K, n_init=4, random_state=0).fit(np.vstack([sift_denso(im) for im in X_tr]))

def bovw(im):
    palabras = vocab.predict(sift_denso(im))
    h = np.bincount(palabras, minlength=K).astype(float)
    return h / h.sum()

reps = {
    'Haralick (4)': lambda im: list(glcm_feats(im).values()),
    'LBP (10)': lbp_hist,
    f'BoVW SIFT (K={K})': bovw,
}
resultados = {}
for nombre, f in reps.items():
    Ftr = np.array([f(im) for im in X_tr]); Fte = np.array([f(im) for im in X_te])
    clf = make_pipeline(StandardScaler(), KNeighborsClassifier(3)).fit(Ftr, y_tr)
    resultados[nombre] = accuracy_score(y_te, clf.predict(Fte))
for k, v in resultados.items():
    print(f'{k:18s} exactitud = {v:.2f}')

**Lectura.** Más complejo no siempre es mejor: con recortes de 64×64, un vocabulario pequeño y descriptores pensados para puntos clave, la bolsa de palabras visuales queda por detrás de LBP y Haralick, que están diseñados precisamente para textura. Elegir el descriptor según el problema es parte del trabajo (y del proyecto).

## 7. Ejercicios

1. **Color.** Repite el experimento de iluminación de la sección 1.3 cambiando el *tono* de la luz (multiplica sólo el canal azul por 1,3). ¿Sigue siendo H el canal más estable? ¿Y a\*/b\*?
2. **Forma.** Añade a la tabla de monedas la *solidez* (`solidity`) y representa área frente a compacidad. ¿Se distinguen grupos de monedas?
3. **Textura.** Calcula la GLCM con distancia `d=4` en lugar de 1. ¿Qué textura cambia más? Relaciónalo con la escala a la que se percibe la textura.
4. **SIFT.** Prueba ángulos de 0° a 180° y escalas de 0,3 a 1,5 y dibuja el porcentaje de correspondencias correctas de SIFT y ORB. ¿Cuál degrada antes?
5. **BoVW.** Cambia K (8, 32, 128) y el tamaño de los recortes (32, 64, 128). ¿Qué ocurre con la exactitud? ¿Y si se recorta de toda la imagen sin separar mitades?